In [1]:
# import libs
import pandas as pd
import numpy as np
import os

# mount google drive
from google.colab import drive
drive.mount('/content/drive')

# create time stamp
from datetime import datetime
now = datetime.now()
dt = now.strftime("%d/%m/%Y %H:%M:%S")
dt_string = str(dt)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# name folder
nm_folder = '/content/drive/Shareddrives/GEM Shared Drive/Programs/Oil & Gas Program/Global Oil and Gas Extraction Tracker (GOGET)/Data sets & reports (O&G)/O&G country materials/United States (GOGET O&G)/New Mexico (GOGET)'
county_folder = nm_folder + '/County Level Data (GOGET)/Location and General Data 2025'
raw_data = county_folder + '/Raw Data'

In [3]:
print(f"Contents of {raw_data}:")
!ls "{raw_data}"

Contents of /content/drive/Shareddrives/GEM Shared Drive/Programs/Oil & Gas Program/Global Oil and Gas Extraction Tracker (GOGET)/Data sets & reports (O&G)/O&G country materials/United States (GOGET O&G)/New Mexico (GOGET)/County Level Data (GOGET)/Location and General Data 2025/Raw Data:
'Bernalillo - Allwells_gp5dxhcaih2wf3p2lqphiovn.csv'
'Catron- Allwells_hvmlry44mknvgn5qtl5wrfv4.csv'
'Chaves - Allwells_d33rq2dfy0b30ofz4uzvf3cw.csv'
'Cibola - Allwells_djkm1t5ghbtwy5t4fb0d2x3x.csv'
'Colfax - Allwells_3zjfe4dfq2oyzaprmq1epfg0.csv'
'Curry - Allwells_pkld3212txyq2nqdimy0so1o.csv'
'De Baca - Allwells_kvoatxcdan1fw5mft3csj5mz.csv'
'Dona Ana - Allwells_ensfbfwdogxltgas0oxaknbi.csv'
'Eddy - Allwells_cfwafupisrpl3ih0ormmc24k.csv'
'Grant - Allwells_u2wxwzcteoo2ijjhb1somvlq.csv'
'Guadalupe - Allwells_1fs4xj1rnalrkpojtqmbxei2.csv'
'Harding - Allwells_aof0habsojzg411fjwpeicon.csv'
'Hidalgo - Allwells_4ltkervdb2bebxjuwau1vpup.csv'
'Lea - Allwells_yfenki1lgdq2rrmfsmo2dwh4.csv'
'Lea - Allwells_yfen

In [4]:
# join all csvs in county_folder
all_counties = pd.concat(map(pd.read_csv, [f"{raw_data}/{file}" for file in os.listdir(raw_data) if file.endswith('.csv')]))

all_counties.columns

/tmp/ipython-input-3040595214.py:2: DtypeWarning: Columns (18,21) have mixed types. Specify dtype option on import or set low_memory=False.
  all_counties = pd.concat(map(pd.read_csv, [f"{raw_data}/{file}" for file in os.listdir(raw_data) if file.endswith('.csv')]))


Index(['api', 'well_name', 'compl_status', 'ocd_ul', 'township', 'range',
       'section', 'sdiv_ul', 'ftg_ns', 'ns_cd', 'ftg_ew', 'ew_cd', 'ogrid_cde',
       'operator', 'property', 'land_type', 'well_type', 'spud_date',
       'plug_date', 'elevgl', 'tvd_depth', 'one_producing_pool_name',
       'last_prod_inj', 'county', 'latitude', 'longitude', 'nbr_compls',
       'acres', ' accum_Oil', ' accm_Gas', ' accm_Water', ' accm_CO2'],
      dtype='object')

In [ ]:
# save all counties to preserve memory
# The original dt_string contains characters like '/' and ' ' which are interpreted as directory separators,
# causing the file path to be treated as a series of non-existent directories.
# To fix this, we will create a file-safe timestamp string.
file_safe_dt_string = now.strftime("%Y-%m-%d_%H-%M-%S")
all_counties.to_csv(f"{county_folder}/all_counties_{file_safe_dt_string}.csv", index=False)

In [ ]:
# skip previous steps and start here once all counties csv is created

# all_counties = pd.read_csv(f"{county_folder}/all_counties_2026-01-09_21-42-01.csv")

# Explicitly create a copy to avoid SettingWithCopyWarning
all_counties_active_well = all_counties.loc[all_counties['compl_status'] == 'Active'].copy()

all_counties_active_well.to_csv(f"{county_folder}/all_counties_active_well_{file_safe_dt_string}.csv", index=False)

In [ ]:
# formation is one_producing_pool_name after the semicolon, split on the semicolon to get pool formation
all_counties_active_well['formation'] = all_counties_active_well['one_producing_pool_name'].str.split(';').str[-1]

# title case all_counties_active_well['formation']
all_counties_active_well['formation'] = all_counties_active_well['formation'].str.title()


In [ ]:
# formation is one_producing_pool_name after the semicolon, split on the semicolon to get pool formation - removing active well filter to look for more matches
all_counties['formation'] = all_counties['one_producing_pool_name'].str.split(';').str[-1]

# title case all_counties_active_well['formation']
all_counties['formation'] = all_counties['formation'].str.title()

In [ ]:
import re

def format_api_standardized(api_raw):
    s_api = str(api_raw)
    cleaned_digits = re.sub(r'\D', '', s_api) # Remove all non-digits

    if not cleaned_digits:
        return "" # Handle cases with no digits

    state_part = cleaned_digits[:2].zfill(2)
    remaining_digits = cleaned_digits[2:]

    county_part = remaining_digits[:3].zfill(3)
    well_part_raw = remaining_digits[3:]

    well_part = well_part_raw.zfill(5)

    # Ensure parts are truncated to their standard lengths
    state_part = state_part[:2]
    county_part = county_part[:3]
    well_part = well_part[:5]

    return f"{state_part}-{county_part}-{well_part}"

# Apply the robust formatting to the 'api' column for all wells
all_counties['api_cleaned'] = all_counties['api'].apply(format_api_standardized)

# Apply the robust formatting to the 'api' column for active wells
all_counties_active_well['api_cleaned'] = all_counties_active_well['api'].apply(format_api_standardized)



In [ ]:
# make new df with just API and formation
api_formation_all = all_counties[['api', 'formation']]
api_formation_active = all_counties_active_well[['api', 'formation']]

api_formation_all.head()


,api,formation
0,3000305000,NaN
1,3000305001,NaN
2,3000305002,NaN
3,3000305007,NaN
4,3000305012,NaN


In [ ]:
# count unique entries in formation - all
api_formation_all['formation'].value_counts()

,count
formation,
Blanco-Mesaverde (Prorated Gas),6458
Basin Fruitland Coal (Gas),4921
Basin Dakota (Prorated Gas),3725
Grayburg-San Andres,2071
San Andres,1245
...,...
"Alamito,South-Gallup (Oil)",1
Mancos,1
Calloway Gallup (Gas),1


In [ ]:
# count unique entries in formation
api_formation_active['formation'].value_counts()

,count
formation,
Blanco-Mesaverde (Prorated Gas),6278
Basin Fruitland Coal (Gas),4239
Basin Dakota (Prorated Gas),3311
Grayburg-San Andres,1783
Blanco P. C. South (Prorated Gas),1033
...,...
"Abo, Mid",1
"Ellenburger, North",1
"Glorieta, Southeast",1


In [ ]:
# check specfic APIs for QC: 3002552244
api_formation_all[api_formation_all['api'] == '3002552244']

,api,formation


In [ ]:
api_formation_all.to_csv(f"{county_folder}/api_formation_all_{file_safe_dt_string}.csv", index=False)
all_counties_active_well.to_csv(f"{county_folder}/all_counties_active_well_{file_safe_dt_string}.csv", index=False)